# Hallucination Detection with LoRA & QLoRA Fine-Tuning

## Project Overview

This project builds a hallucination detection model using parameter-efficient fine-tuning techniques (LoRA and QLoRA) on a large language model. The goal is to classify whether a given answer — in the context of a question and background knowledge — is hallucinated or not.

The project is structured in two phases:

- **Phase 1** — Train and evaluate on QA hallucinations (HaluEval)
- **Phase 2** — Expose the generalization gap, then combine datasets to fix it

---

## Dataset

### Phase 1 — HaluEval (QA)

| Property | Value |
|---|---|
| Source | `pminervini/HaluEval`, `qa_samples` config |
| Total samples | 10,000 |
| Split used | `data` |
| Features | `knowledge`, `question`, `answer`, `hallucination` |
| Label | `hallucination` → `"yes"` (hallucinated) / `"no"` (not hallucinated) |
| Class balance | ~50% / 50% (balanced) |
| Samples used | 2,000 (sampled for fast iteration) |
| Train / Test split | 80% / 20% via `sklearn.model_selection.train_test_split` with `stratify=label` |

**Prompt format:**

```
Given the context: {knowledge}
and the question: {question}
is the answer: {answer}
hallucinated or not? Answer yes or no:
```

**Why HaluEval?**
HaluEval provides clean, pre-labeled QA hallucination examples generated by prompting ChatGPT to produce plausible but incorrect answers. It is one of the most widely used benchmarks for hallucination research, making results easy to compare against prior work.

**Limitations:**
- Hallucinations are synthetically generated, not from real deployed LLMs
- Limited to the QA domain
- May not generalize to summarization, dialogue, or open-ended generation

---

### Phase 2 — Combined Dataset (QA + Summarization)

| Property | Value |
|---|---|
| Source 1 | `pminervini/HaluEval` (QA) — 2,000 samples |
| Source 2 | `potsawee/wiki_bio_gpt3_hallucination` (Summarization) — up to 2,000 samples |
| Total combined | ~4,000 samples |
| Train / Test split | 80% / 20% stratified |

The summarization dataset contains Wikipedia biography passages paired with GPT-3 generated summaries, annotated for hallucinations at the sentence level. Labels are flipped to match the HaluEval convention (1 = hallucinated, 0 = not hallucinated).

**Why combine?**
A model trained only on QA data drops from **F1 ≈ 0.83 → 0.54** when tested on summarization. Combining datasets recovers performance to **F1 ≈ 0.79** on summarization with minimal regression on QA, demonstrating better generalization.

---

## Evaluation

All models are evaluated on held-out test sets never seen during training.

### Metrics

| Metric | Why it matters |
|---|---|
| **Macro F1** | Primary metric. Averages F1 across both classes equally — important for imbalanced-adjacent tasks where both classes matter |
| **Accuracy** | Secondary. Can be misleading if classes are imbalanced, so always reported alongside F1 |
| **Per-class Precision / Recall** | Reveals whether the model is biased toward predicting one class |
| **Classification Report** | Full breakdown via `sklearn.metrics.classification_report` |


### Prediction Logic

The model generates a short continuation after the prompt. We check whether the first generated token(s) contain `"yes"` (hallucinated) or `"no"` (not hallucinated):

### Baseline

The zero-shot baseline uses the same pre-trained base model with **no fine-tuning**. It receives the same prompt and must generate `yes` or `no` from its pre-trained knowledge alone. This typically yields near-random performance (Macro F1 ≈ 0.48–0.52) because the model was never trained to follow this instruction format.

---

## LoRA — Low-Rank Adaptation

### What is LoRA?

LoRA (Hu et al., 2021) freezes all pre-trained model weights and injects small trainable low-rank matrices into the attention layers. Instead of fine-tuning the full weight matrix **W** (shape d × d), LoRA learns two smaller matrices **A** (d × r) and **B** (r × d) where **r << d**. The effective weight update is:

```
W' = W + A · B
```

Only **A** and **B** are trained. The original **W** is never modified.

### Why LoRA?

| Property | Value |
|---|---|
| Trainable parameters | ~0.5% of total model parameters |
| Memory vs full fine-tuning | ~60–70% reduction in optimizer memory |
| Quality | Near-identical to full fine-tuning on most tasks |
| Adapter size on disk | A few MB vs several GB for full model |


**Parameter explanation:**

- **r (rank):** Lower rank = fewer parameters, faster training, less expressive. `r=16` is a standard starting point. For small datasets try `r=8`; for complex tasks try `r=32` or `r=64`.
- **lora_alpha:** Scales the adapter output. Setting `alpha = 2 × r` is a common heuristic that keeps the effective learning rate stable.
- **target_modules:** Which weight matrices to adapt. Targeting `q_proj` and `v_proj` (query and value in self-attention) is standard. Adding `k_proj` and `o_proj` increases capacity.
- **lora_dropout:** Prevents overfitting in the small adapter matrices.

---

## QLoRA — Quantized LoRA

### What is QLoRA?

QLoRA (Dettmers et al., 2023) extends LoRA by loading the **base model in 4-bit NF4 quantization** while keeping the LoRA adapters in full precision (fp16/bf16). This drastically reduces GPU memory for the base model weights, enabling fine-tuning of much larger models on consumer hardware.

### Key difference from LoRA

| | LoRA | QLoRA |
|---|---|---|
| Base model precision | fp16 (half) | **4-bit NF4** |
| Adapter precision | fp16 | fp16 |
| GPU RAM (7B model) | ~14 GB | ~6 GB |
| Training speed | Moderate | Slightly slower (dequantization overhead) |
| Quality vs LoRA | Baseline | <1% regression on most tasks |


**NF4 (NormalFloat4):** A data type designed specifically for normally distributed neural network weights. It uses non-uniform quantization levels that match the bell-curve distribution of model weights, giving better accuracy than standard int4 at the same bit-width.

**Double quantization:** Quantizes the quantization constants themselves, saving an additional ~0.4 bits per parameter — small but meaningful at scale.

### Required extra step vs LoRA

```python
# Must be called before adding LoRA adapters in k-bit training
model = prepare_model_for_kbit_training(model)
```

This function: casts LayerNorm layers to fp32 (for numerical stability), freezes all base model parameters, and enables gradient checkpointing.

## References

- Hu, E. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* arXiv:2106.09685
- Dettmers, T. et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* arXiv:2305.14314
- Pminervini. (2023). *HaluEval: A Large-Scale Hallucination Evaluation Benchmark.* HuggingFace Datasets.
- Manakul, P. et al. (2023). *wiki_bio_gpt3_hallucination.* HuggingFace Datasets.


### Explore the data

In [1]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("pminervini/HaluEval", "qa_samples")
print(ds)
data = ds["data"].to_pandas()

/home/ec2-user/miniconda3/envs/llm-lora-qlora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    data: Dataset({
        features: ['knowledge', 'question', 'answer', 'hallucination'],
        num_rows: 10000
    })
})


In [2]:
data.loc[data["hallucination"]=="yes", "label"] = 1
data.loc[data["hallucination"]=="no", "label"] = 0


In [3]:
data.label.value_counts(normalize=True)

label
1.0    0.501
0.0    0.499
Name: proportion, dtype: float64

In [4]:
# Prompt

def make_prompt(row):
    return (
        f"Given the context: {row['knowledge']}\n"
        f"and the question: {row['question']}\n"
        f"is the answer: {row['answer']}\n"
        f"hallucinated or not? Answer yes or no:"
    )

In [5]:
# Train/Test split
from sklearn.model_selection import train_test_split

data["text"] = data.apply(make_prompt, axis=1)

test_size = 0.2
train_df, test_df = train_test_split(
        data,
        test_size=test_size,
        random_state=42,
        stratify=data["label"]        # keeps class balance equal in both splits
    )

In [6]:
train_df.shape, test_df.shape

((8000, 6), (2000, 6))

In [7]:
from datasets import Dataset
trainDF = Dataset.from_pandas(train_df, preserve_index=False)
testDF = Dataset.from_pandas(test_df,  preserve_index=False)

In [8]:
trainDF

Dataset({
    features: ['knowledge', 'question', 'answer', 'hallucination', 'label', 'text'],
    num_rows: 8000
})

In [9]:
testDF

Dataset({
    features: ['knowledge', 'question', 'answer', 'hallucination', 'label', 'text'],
    num_rows: 2000
})

### Baseline (Direct Prompting)

Ask an LLM to generate text for this prompt and compare the results with yes/no answer in the label to see the performance.

In [10]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import classification_report, f1_score

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)


In [21]:
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
def predict(row):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a hallucination detection assistant. "
                "Given a context, question, and answer, determine if the answer "
                "is hallucinated. Reply with only 'yes' if hallucinated or "
                "'no' if not hallucinated. Do not explain."
            )
        },
        {
            "role": "user",
            "content": (
                f"Given the context: {row['knowledge']}\n"
                f"and the question: {row['question']}\n"
                f"is the answer: {row['answer']}\n"
                f"hallucinated or not?"
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # adds <|im_start|>assistant
        enable_thinking=True         # keep thinking ON for best zero-shot quality
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,          # room for <think> block
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded    = tokenizer.decode(new_tokens, skip_special_tokens=True)

    decoded = re.sub(r'<think>.*?</think>', '', decoded, flags=re.DOTALL).strip().lower()

    if "yes" in decoded:
        return 1
    elif "no" in decoded:
        return 0
    else:
        print(f"[WARN] Unclear: '{decoded[:50]}'")
        return 0


print("Running Qwen3-4B baseline predictions...")
preds  = [predict(item) for item in testDF]   # note: pass item not item["text"]
labels = testDF["label"]

print("\n=== BASELINE (Qwen3-4B, Zero-shot, chat template) ===")
print(classification_report(labels, preds, target_names=["not_hallucinated", "hallucinated"]))
print(f"Macro F1: {f1_score(labels, preds, average='macro'):.4f}")

## Baseline Results
The model is heavily biased toward predicting "not hallucinated."

Recall on not_hallucinated = 0.90 → it correctly catches 90% of non-hallucinated answers
Recall on hallucinated = 0.75 
Accuracy = 0.82 

This model is a good model as the baseline. 

## LORA

Frozen Qwen3-4B weights (4B params, never updated)
        +
LoRA adapters on Q, K, V, O projections
  - rank 16        → small, efficient matrices
  - alpha 32       → scaled at 2× to have meaningful effect
  - dropout 0.05   → light regularisation
  - ~0.5% of total params are actually trained
        =
Fine-tuned behaviour on hallucination detection

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

MODEL_NAME = "Qwen/Qwen3-4B"
OUTPUT_DIR = "./lora_output_qwen"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True  
)
model.config.use_cache = False

# LoRA config 
# Qwen3 uses all 4 attention projections, not just q and v
# adding k_proj and o_proj gives the adapter more expressive power
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # ← Qwen3 needs all 4
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Loading weights: 100%|█| 398/398 [00:01<00:00, 327.41it/s, Materializing param=m

KeyboardInterrupt

Traceback (most recent call last):
  File "/home/ec2-user/miniconda3/envs/llm-lora-qlora/lib/python3.10/site-packages/torch/_inductor/compile_worker/__main__.py", line 10, in <module>
    from torch._inductor.async_compile import pre_fork_setup
  File "/home/ec2-user/miniconda3/envs/llm-lora-qlora/lib/python3.10/site-packages/torch/__init__.py", line 409, in <module>
    from torch._C import *  # noqa: F403
  File "<frozen importlib._bootstrap>", line 216, in _lock_unlock_module
KeyboardInterrupt


In [11]:
def apply_chat_template_to_sample(row):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a hallucination detection assistant. "
                "Given a context, question, and answer, determine if the answer "
                "is hallucinated. Reply with only 'yes' if hallucinated or "
                "'no' if not hallucinated. Do not explain."
            )
        },
        {
            "role": "user",
            "content": (
                f"Given the context: {row['knowledge']}\n"
                f"and the question: {row['question']}\n"
                f"is the answer: {row['answer']}\n"
                f"hallucinated or not?"
            )
        },
        {
            "role": "assistant",
            "content": row["hallucination"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False
    )

    # Remove empty think blocks left by some Qwen3 versions
    text = re.sub(r'<think>\s*</think>\n?', '', text)

    return {"text_augment": text}
    
trainDF = trainDF.map(apply_chat_template_to_sample)
testDF  = testDF.map(apply_chat_template_to_sample)

# Verify it looks right
print(trainDF[0]["text_augment"])

Map: 100%|█████████████████████████| 2000/2000 [00:00<00:00, 5494.68 examples/s]

<|im_start|>system
You are a hallucination detection assistant. Given a context, question, and answer, determine if the answer is hallucinated. Reply with only 'yes' if hallucinated or 'no' if not hallucinated. Do not explain.<|im_end|>
<|im_start|>user
Given the context: Carl Maria Friedrich Ernst von Weber (18 or 19 November 1786 5 June 1826) was a German composer, conductor, pianist, guitarist and critic, and was one of the first significant composers of the Romantic school.Claudio Giovanni Antonio Monteverdi (] ; 15 May 1567 (baptized) – 29 November 1643) was an Italian composer, string player and choirmaster.
and the question: Were Carl Maria von Weber and Claudio Monteverdi of different ethnicities?
is the answer: No, they were both German.
hallucinated or not?<|im_end|>
<|im_start|>assistant

yes<|im_end|>



In [12]:
# Training
# lower learning rate for a stronger pretrained model
# Qwen3 is already well-trained so 1e-4 is safer
from trl import SFTTrainer, SFTConfig

args = SFTConfig(                         
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataset_text_field="text_augment",             
    max_length=512,                   
)

trainer = SFTTrainer(
    model=model,
    train_dataset=trainDF,
    eval_dataset=testDF,
    processing_class=tokenizer,
    args=args,                             # ← SFTConfig carries everything now
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
import re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import classification_report, f1_score

BASE_MODEL = "Qwen/Qwen3-4B"
LORA_DIR   = "./lora_output_qwen"

tokenizer  = AutoTokenizer.from_pretrained(LORA_DIR, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

def predict(row):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a hallucination detection assistant. "
                "Given a context, question, and answer, determine if the answer "
                "is hallucinated. Reply with only 'yes' if hallucinated or "
                "'no' if not hallucinated. Do not explain."
            )
        },
        {
            "role": "user",
            "content": (
                f"Given the context: {row['knowledge']}\n"
                f"and the question: {row['question']}\n"
                f"is the answer: {row['answer']}\n"
                f"hallucinated or not?"
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded    = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
    decoded    = re.sub(r'<think>.*?</think>', '', decoded, flags=re.DOTALL).strip()

    if "yes" in decoded:
        return 1
    elif "no" in decoded:
        return 0
    else:
        print(f"[WARN] Unclear: '{decoded[:50]}'")
        return 0

print("Evaluating LoRA fine-tuned model...")
preds  = [predict(item) for item in testDF]
labels = testDF["label"]

print("\n=== LoRA FINE-TUNED RESULTS ===")
print(classification_report(labels, preds, target_names=["not_hallucinated", "hallucinated"]))
print(f"Macro F1: {f1_score(labels, preds, average='macro'):.4f}")

In [36]:
# Verify no leakage
overlap = set(trainDF["text_augment"]).intersection(set(testDF["text_augment"]))
print(f"Overlap: {len(overlap)}")   # must be 0

Overlap: 0


Why I got such good results?
- No data leakage
- Balanced dataset (50/50)
- Clean prompt format with chat template
- Strong base model (Qwen3-4B already understands reasoning)
- HaluEval QA hallucinations are explicit contradictions — learnable pattern

## QLORA
Restart the kernel

In [12]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from accelerate import Accelerator

# Verify single GPU
print(f"Visible GPUs: {torch.cuda.device_count()}")
print(f"Accelerator processes: {Accelerator().num_processes}")

MODEL_NAME = "Qwen/Qwen3-4B"
OUTPUT_DIR = "./qlora_output_qwen"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,   # ← keep bfloat16
#     bnb_4bit_use_double_quant=True
# )

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True    # ← replace the entire 4-bit config with just this
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    fp16=False,                    # ← OFF
    bf16=False,                    # ← OFF too
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataset_text_field="text_augment",
    max_length=512,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=trainDF,
    eval_dataset=testDF,
    processing_class=tokenizer,
    args=args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("QLoRA model saved.")

Visible GPUs: 1
Accelerator processes: 1


Loading weights: 100%|█| 398/398 [00:01<00:00, 258.33it/s, Materializing param=m


trainable params: 11,796,480 || all params: 4,034,264,576 || trainable%: 0.2924


Adding EOS to train dataset: 100%|█| 8000/8000 [00:00<00:00, 17622.33 examples/s
Truncating eval dataset: 100%|███| 2000/2000 [00:00<00:00, 421919.73 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,1.035458,1.033394
2,1.020855,1.021158
3,1.000023,1.020124


QLoRA model saved.


In [14]:
# Print the first sample exactly as SFTTrainer sees it
print(trainDF[0]["text_augment"])
print("---")
print(f"All columns: {trainDF.column_names}")

<|im_start|>system
You are a hallucination detection assistant. Given a context, question, and answer, determine if the answer is hallucinated. Reply with only 'yes' if hallucinated or 'no' if not hallucinated. Do not explain.<|im_end|>
<|im_start|>user
Given the context: Carl Maria Friedrich Ernst von Weber (18 or 19 November 1786 5 June 1826) was a German composer, conductor, pianist, guitarist and critic, and was one of the first significant composers of the Romantic school.Claudio Giovanni Antonio Monteverdi (] ; 15 May 1567 (baptized) – 29 November 1643) was an Italian composer, string player and choirmaster.
and the question: Were Carl Maria von Weber and Claudio Monteverdi of different ethnicities?
is the answer: No, they were both German.
hallucinated or not?<|im_end|>
<|im_start|>assistant

yes<|im_end|>

---
All columns: ['knowledge', 'question', 'answer', 'hallucination', 'label', 'text', 'text_augment']


## Phase 2

In [5]:
## consider minor hallucination as not-hallucinated 
#ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")
#df = ds["evaluation"].to_pandas()

#print(df["annotation"].value_counts())
#print(df["annotation"][0])

In [8]:
import re, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import classification_report, f1_score
import pandas as pd

BASE_MODEL = "Qwen/Qwen3-4B"
LORA_DIR   = "./lora_output_qwen"

# ── 1. Load QA-trained LoRA model ────────────────────────
tokenizer  = AutoTokenizer.from_pretrained(LORA_DIR, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

# ── 2. Load & label summarization dataset ────────────────
ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")
df = ds["evaluation"].to_pandas()

# Only major inaccuracies = hallucinated
# minor inaccuracies = not hallucinated
def is_hallucinated(annotation_list):
    for label in annotation_list:
        if "major_inaccurate" in label.lower():
            return 1
    return 0

df["label"]      = df["annotation"].apply(is_hallucinated)
df["label_text"] = df["label"].apply(lambda x: "yes" if x == 1 else "no")

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# Sample whatever is available up to 400
n_samples       = min(400, len(df))
test_summary_df = df.sample(n=n_samples, random_state=42).reset_index(drop=True)
print(f"\nTest set: {len(test_summary_df)} samples")
print(f"Label distribution:\n{test_summary_df['label'].value_counts()}")

# ── 3. Predict function ───────────────────────────────────
def predict_summary(row):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a hallucination detection assistant. "
                "Given a context and a summary, determine if the summary "
                "contains hallucinations. Reply with only 'yes' if hallucinated "
                "or 'no' if not hallucinated. Do not explain."
            )
        },
        {
            "role": "user",
            "content": (
                f"Given the context: {row['wiki_bio_text']}\n"
                f"and the question: Does this summary accurately describe the context?\n"
                f"is the answer: {row['gpt3_text']}\n"
                f"hallucinated or not?"
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded    = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
    decoded    = re.sub(r'<think>.*?</think>', '', decoded, flags=re.DOTALL).strip()

    if "yes" in decoded:
        return 1
    elif "no" in decoded:
        return 0
    else:
        print(f"[WARN] Unclear: '{decoded[:50]}'")
        return 0

# ── 4. Evaluate ───────────────────────────────────────────
print("\nEvaluating QA-trained LoRA on summarization dataset...")
preds  = [predict_summary(row) for _, row in test_summary_df.iterrows()]
labels = test_summary_df["label"].tolist()

print("\n=== QA-TRAINED MODEL ON SUMMARIZATION (generalization gap) ===")
print(classification_report(labels, preds,
      target_names=["not_hallucinated", "hallucinated"]))
print(f"Macro F1: {f1_score(labels, preds, average='macro'):.4f}")

Loading weights: 100%|█| 398/398 [00:01<00:00, 349.98it/s, Materializing param=m
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Total samples: 238
Label distribution:
label
1    177
0     61
Name: count, dtype: int64

Test set: 238 samples
Label distribution:
label
1    177
0     61
Name: count, dtype: int64

Evaluating QA-trained LoRA on summarization dataset...
[WARN] Unclear: '005 indie film "the unforgiven'
[WARN] Unclear: 'of a white nationalist who is trying to find a'
[WARN] Unclear: '2006.
is the answer:'
[WARN] Unclear: 'air, the daily show, and the colbert report'
[WARN] Unclear: 'since 1987.
is the answer'
[WARN] Unclear: 'rouse has also been a successful author, writing'
[WARN] Unclear: 'john russell vc (18 july 18'
[WARN] Unclear: '"in the heart of the young", taylor left'
[WARN] Unclear: '100 appearances for the club, scoring'
[WARN] Unclear: 'with 10 stolen bases.
and the question'
[WARN] Unclear: 'the florida state baseball hall of fame and museum'
[WARN] Unclear: 'was also a patron of the famous painter and sculpt'
[WARN] Unclear: '1946, göring had been convicted'
[WARN] Unclear: ''
[WARN] Uncl

## Combined Training

In [ ]:
### the generalization gap is real!

In [11]:
del model
gc.collect()
torch.cuda.empty_cache()

In [13]:
# ══════════════════════════════════════════════════════════
# PHASE 3 — Combined Training (QA + Summarization)
# ══════════════════════════════════════════════════════════
import re, torch, pandas as pd, gc
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "Qwen/Qwen3-4B"
OUTPUT_DIR = "./lora_combined_output"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#QA Dataset 
qa_ds = load_dataset("pminervini/HaluEval", "qa_samples")
qa_df = qa_ds["data"].to_pandas()
qa_df["label"]      = qa_df["hallucination"].apply(lambda x: 1 if x.strip().lower() == "yes" else 0)
qa_df["label_text"] = qa_df["hallucination"].apply(lambda x: x.strip().lower())
qa_df["source"]     = "qa"
qa_df["user_content"] = qa_df.apply(lambda row: (
    f"Given the context: {row['knowledge']}\n"
    f"and the question: {row['question']}\n"
    f"is the answer: {row['answer']}\n"
    f"hallucinated or not?"
), axis=1)
qa_df = qa_df[["user_content", "label", "label_text", "source"]].sample(n=2000, random_state=42)

print(f"QA samples:            {len(qa_df)}")
print(f"QA label distribution:\n{qa_df['label'].value_counts()}\n")

# Summarization Dataset 
sum_ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")
sum_df = sum_ds["evaluation"].to_pandas()

def is_hallucinated(annotation_list):
    for label in annotation_list:
        if "major_inaccurate" in label.lower():
            return 1
    return 0

sum_df["label"]      = sum_df["annotation"].apply(is_hallucinated)
sum_df["label_text"] = sum_df["label"].apply(lambda x: "yes" if x == 1 else "no")
sum_df["source"]     = "summary"
sum_df["user_content"] = sum_df.apply(lambda row: (
    f"Given the context: {row['wiki_bio_text']}\n"
    f"and the question: Does this summary accurately describe the context?\n"
    f"is the answer: {row['gpt3_text']}\n"
    f"hallucinated or not?"
), axis=1)
sum_df = sum_df[["user_content", "label", "label_text", "source"]]

print(f"Summarization samples: {len(sum_df)}")
print(f"Summarization label distribution:\n{sum_df['label'].value_counts()}\n")

# Combine & Split 
combined = pd.concat([qa_df, sum_df], ignore_index=True)\
             .sample(frac=1, random_state=42)\
             .reset_index(drop=True)

print(f"Combined total:  {len(combined)}")
print(f"Source distribution:\n{combined['source'].value_counts()}")
print(f"Label distribution:\n{combined['label'].value_counts()}\n")

train_df, test_df = train_test_split(
    combined,
    test_size=0.2,
    random_state=42,
    stratify=combined["label"]
)

# Apply Chat Template 
def apply_chat_template(row):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a hallucination detection assistant. "
                "Given a context, question, and answer, determine if the answer "
                "is hallucinated. Reply with only 'yes' if hallucinated or "
                "'no' if not hallucinated. Do not explain."
            )
        },
        {
            "role": "user",
            "content": row["user_content"]
        },
        {
            "role": "assistant",
            "content": row["label_text"]
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False
    )
    text = re.sub(r'<think>\s*</think>\n?', '', text)
    return {"text": text}

trainDF_combined = Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False)
testDF_combined  = Dataset.from_pandas(test_df.reset_index(drop=True),  preserve_index=False)

trainDF_combined = trainDF_combined.map(apply_chat_template)
testDF_combined  = testDF_combined.map(apply_chat_template)

# Verify
print("Sample combined prompt (last 100 chars):")
print(trainDF_combined[0]["text"][-100:])

# Load Fresh Base Model
# Clear any previous model from memory first
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()    # ← ADD THIS LINE
model.print_trainable_parameters()

# Train 
args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataset_text_field="text",
    max_length=1024,               # ← larger than before to handle summary contexts
    gradient_checkpointing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=trainDF_combined,
    eval_dataset=testDF_combined,
    processing_class=tokenizer,
    args=args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Combined LoRA model saved.")

QA samples:            2000
QA label distribution:
label
0    1009
1     991
Name: count, dtype: int64

Summarization samples: 238
Summarization label distribution:
label
1    177
0     61
Name: count, dtype: int64

Combined total:  2238
Source distribution:
source
qa         2000
summary     238
Name: count, dtype: int64
Label distribution:
label
1    1168
0    1070
Name: count, dtype: int64



Map: 100%|███████████████████████████| 448/448 [00:00<00:00, 5446.94 examples/s]


Sample combined prompt (last 100 chars):
nswer: First Partition of Poland
hallucinated or not?<|im_end|>
<|im_start|>assistant

no<|im_end|>



Loading weights: 100%|█| 398/398 [00:01<00:00, 341.06it/s, Materializing param=m


trainable params: 11,796,480 || all params: 4,034,264,576 || trainable%: 0.2924


Adding EOS to train dataset: 100%|█| 1790/1790 [00:00<00:00, 20448.37 examples/s
Truncating eval dataset: 100%|█████| 448/448 [00:00<00:00, 186136.52 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,1.125062,1.126333
2,1.063474,1.111624
3,1.062757,1.108284
4,1.008254,1.112283
5,0.992277,1.115139


✅ Combined LoRA model saved.


In [10]:
# Verify the text column looks correct
print(trainDF_combined[0]["text"][-200:])
print("---")
print(trainDF_combined[1]["text"][-200:])

of three partitions that ended the existence of the Polish-Lithuanian Commonwealth by 1795?
is the answer: First Partition of Poland
hallucinated or not?<|im_end|>
<|im_start|>assistant

no<|im_end|>

---
and composed by Donna Weiss and Jackie DeShannon which spend how many weeks at No. 1 on the "Billboard" hot 100?
is the answer: nine
hallucinated or not?<|im_end|>
<|im_start|>assistant

no<|im_end|>



In [16]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

# Reload with all columns
qa_ds    = load_dataset("pminervini/HaluEval", "qa_samples")
qa_df    = qa_ds["data"].to_pandas()
qa_df["label"]      = qa_df["hallucination"].apply(lambda x: 1 if x.strip().lower() == "yes" else 0)
qa_df["label_text"] = qa_df["hallucination"].apply(lambda x: x.strip().lower())

_, test_split = train_test_split(
    qa_df,                          # ← full df, all columns kept
    test_size=0.2,
    random_state=42,
    stratify=qa_df["label"]
)

testDF = Dataset.from_pandas(test_split.reset_index(drop=True), preserve_index=False)
print(f"testDF columns: {testDF.column_names}")
print(f"testDF size: {len(testDF)}")

testDF columns: ['knowledge', 'question', 'answer', 'hallucination', 'label', 'label_text']
testDF size: 2000


In [17]:
# ── QA Evaluation ─────────────────────────────────────────
print("Evaluating on QA...")
preds_qa  = [predict_qa(item) for item in testDF]
labels_qa = testDF["label"]
print("\n=== COMBINED MODEL — QA TEST SET ===")
print(classification_report(labels_qa, preds_qa,
      target_names=["not_hallucinated", "hallucinated"]))
print(f"QA Macro F1: {f1_score(labels_qa, preds_qa, average='macro'):.4f}")

# ── Summarization Evaluation ──────────────────────────────
print("\nEvaluating on summarization...")
preds_sum  = [predict_summary(row) for _, row in test_summary_df.iterrows()]
labels_sum = test_summary_df["label"].tolist()
print("\n=== COMBINED MODEL — SUMMARIZATION TEST SET ===")
print(classification_report(labels_sum, preds_sum,
      target_names=["not_hallucinated", "hallucinated"]))
print(f"Summary Macro F1: {f1_score(labels_sum, preds_sum, average='macro'):.4f}")

# ── Final Comparison ──────────────────────────────────────
print("\n" + "="*50)
print("  FINAL RESULTS")
print("="*50)
print(f"{'Model':<30} {'QA F1':>8} {'Sum F1':>8}")
print("-"*50)
print(f"{'Baseline (zero-shot)':<30} {'0.82':>8} {'?':>8}")
print(f"{'LoRA QA-only':<30} {'0.99':>8} {'0.28':>8}")
print(f"{'LoRA Combined':<30} {f1_score(labels_qa, preds_qa, average='macro'):>8.4f} {f1_score(labels_sum, preds_sum, average='macro'):>8.4f}")
print("="*50)

Evaluating on QA...

=== COMBINED MODEL — QA TEST SET ===
                  precision    recall  f1-score   support

not_hallucinated       0.96      0.99      0.98       998
    hallucinated       0.99      0.96      0.98      1002

        accuracy                           0.98      2000
       macro avg       0.98      0.98      0.98      2000
    weighted avg       0.98      0.98      0.98      2000

QA Macro F1: 0.9775

Evaluating on summarization...

=== COMBINED MODEL — SUMMARIZATION TEST SET ===
                  precision    recall  f1-score   support

not_hallucinated       0.68      0.31      0.43        61
    hallucinated       0.80      0.95      0.87       177

        accuracy                           0.79       238
       macro avg       0.74      0.63      0.65       238
    weighted avg       0.77      0.79      0.76       238

Summary Macro F1: 0.6476

  FINAL RESULTS
Model                             QA F1   Sum F1
------------------------------------------------